In [1]:
import ast
import glob
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
for genus_name in keep_genus:
    basic_dir = rf'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    anno_dir = rf'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/annotations'

    replicon_data = pd.read_csv(f'{basic_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_data = replicon_data[replicon_data['ms-label'] == 'NMS_replicon']
    replicon_data = replicon_data[['accession', 'size', 'average plasmid fraction-pident_90', 'category-pident_90', 'cp-label']]
    replicon_data = replicon_data.rename(columns={'accession': 'contig_name', 'cp-label': 'initial_type'})
    
    plasflow_re = pd.read_csv(f'{anno_dir}/PlasFlow_NMS_replicon_result.txt', sep='\t')
    plasflow_re['plasflow_pdc'] = plasflow_re['label'].str.split('.').str[0]
    prediction_re = pd.merge(replicon_data, plasflow_re[['contig_name', 'plasflow_pdc']], on='contig_name', how='left')
    
    plasmer_re = pd.read_csv(f'{anno_dir}/results/Plasmer_NMS_replicon_result.plasmer.predClass.tsv', sep='\t', header=None)
    plasmer_re.rename(columns={0: 'contig_name', 1:'plasmer_pdc'}, inplace=True)
    prediction_re = pd.merge(prediction_re, plasmer_re[['contig_name', 'plasmer_pdc']], on='contig_name', how='left')
    
    rfplasmid_re = pd.read_csv(f'{anno_dir}/rfplamsid/prediction.csv')
    rfplasmid_re['contig_name'] = rfplasmid_re['contigID'].str.split(' ').str[0]
    rfplasmid_re['rfplasmid_pdc'] = rfplasmid_re['prediction'].map({'p': 'plasmid', 'c': 'chromosome'}).fillna('unclassified')
    prediction_re = pd.merge(prediction_re, rfplasmid_re[['contig_name', 'rfplasmid_pdc']], on='contig_name', how='left')
    
    pattern = f"{anno_dir}/Deeplamsid/outPR.*/predictions.txt"
    found_files = glob.glob(pattern, recursive=False)
    
    file_path = found_files[0]
    new_path = file_path.replace('txt', 'tsv')
    
    with open(file_path, 'r') as f:
        content = f.read().replace('name,pred,conf', 'name\tpred\tconf')
        content = content.replace(',PLASMID,', '\tPLASMID\t')
        content = content.replace(',GENOME,', '\tGENOME\t')
        content = content.replace(',SHORTER_1000.0,', '\tSHORTER_1000.0\t')
        content = content.replace(',LONGER_330000.0,', '\tLONGER_330000.0\t')
        content = content.replace(',NOFEATURE,', '\tNOFEATURE\t')
    
    with open(new_path, 'w+') as f:
        f.write(content)
    
    deeplasmid_re = pd.read_csv(new_path, sep='\t')
    deeplasmid_re['contig_name'] = deeplasmid_re['name'].str.split(' ').str[0].str.upper()
    deeplasmid_re['deeplasmid_pdc'] = deeplasmid_re['pred'].map({'PLASMID': 'plasmid', 'GENOME': 'chromosome'}).fillna('unclassified')
    prediction_re = pd.merge(prediction_re, deeplasmid_re[['contig_name', 'deeplasmid_pdc']], on='contig_name', how='left')

    #print(prediction_re)

    prediction_re.to_csv(f"{anno_dir}/NMS_replicon_type_prediction_results.csv", index=False)